<a href="https://colab.research.google.com/github/Subhash-2910/flyrank-ML-T1/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Subhash-2910/flyrank-ML-T1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Subhash-2910/flyrank-ML-T1.git"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )
else:
    # Find the repo root from wherever this kernel started.
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv")
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



## Finding 1: AI sessions measure direct click-through visits

The FlyRank research paper defines AI sessions as direct visits from AI tools to tracked blog pages.

**Methodology question:** How is an AI-tool referral identified, and what kinds of AI visibility are excluded? Direct sessions are not the same as citations, impressions, or zero-click exposure inside AI tools. I would interpret this finding as measured direct AI-referred traffic, not total AI-search visibility.

## Finding 2: Search volume alone does not explain observed search visibility

The paper reports that search demand alone does not determine whether a page receives impressions.

**Methodology question:** How were search volume and impressions aligned across content types, clients, search intent, and ranking positions? I would want to know whether the analysis checks these possible differences before making a broad claim about all content.

These questions are constructive. They help define the scope of the paper’s measured findings rather than trying to disprove them.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Pseudonymized clients:", df["client_id"].nunique())
print("Columns:", len(df.columns))

# A simple descriptive check related to the paper's search-volume finding.
search_volume_impressions_corr = df["search_volume"].corr(df["impressions_90d"])

print(
    "Observed correlation between search_volume and impressions_90d:",
    round(search_volume_impressions_corr, 4)
)

Rows: 30000
Pseudonymized clients: 32
Columns: 44
Observed correlation between search_volume and impressions_90d: 0.0012


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


I compare a random page-level split with a client-grouped split.

The random split can place pages from the same client in both training and test data. This can make results look stronger because client-specific patterns may be shared across both sets.

The grouped split is more honest because it tests whether the model can generalize to pages from clients it did not train on.

My target is whether the observed `trend_direction` is `down`.

For this audit, I use only stable content and market features that are not derived from the 30-day trend comparison windows. I exclude `trend_pct`, all last/previous-30-day columns, IDs, and 90-day performance totals because they can overlap with the measured trend period.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

id_col = "content_id"
group_col = "client_id"
target_col = "decline_label"

# Outcome: observed downward trend.
df[target_col] = (df["trend_direction"] == "down").astype(int)

# Strict decision-time-safe feature set for this audit.
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
]

feature_cols = numeric_features + categorical_features

model_df = df[[id_col, group_col, target_col] + feature_cols].copy()

def make_model():
    preprocessor = ColumnTransformer([
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ])

def precision_at_k(y_true, scores, k=20):
    ranked = pd.DataFrame({
        "label": y_true.to_numpy(),
        "score": scores,
    }).nlargest(k, "score")
    return ranked["label"].mean()

# BEFORE: random page-level split.
random_train, random_test = train_test_split(
    model_df,
    test_size=0.20,
    random_state=42,
    stratify=model_df[target_col],
)

random_model = make_model()
random_model.fit(random_train[feature_cols], random_train[target_col])
random_scores = random_model.predict_proba(random_test[feature_cols])[:, 1]

# AFTER: client-grouped split.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

group_train_idx, group_test_idx = next(
    splitter.split(
        model_df,
        y=model_df[target_col],
        groups=model_df[group_col],
    )
)

group_train = model_df.iloc[group_train_idx].copy()
group_test = model_df.iloc[group_test_idx].copy()

assert set(group_train[group_col]).isdisjoint(set(group_test[group_col]))

group_model = make_model()
group_model.fit(group_train[feature_cols], group_train[target_col])
group_scores = group_model.predict_proba(group_test[feature_cols])[:, 1]

before_after = pd.DataFrame({
    "validation_design": [
        "Random page-level split",
        "Client-grouped split",
    ],
    "precision_at_20": [
        precision_at_k(random_test[target_col], random_scores),
        precision_at_k(group_test[target_col], group_scores),
    ],
    "average_precision": [
        average_precision_score(random_test[target_col], random_scores),
        average_precision_score(group_test[target_col], group_scores),
    ],
    "roc_auc": [
        roc_auc_score(random_test[target_col], random_scores),
        roc_auc_score(group_test[target_col], group_scores),
    ],
}).round(3)

display(before_after)

print("Training clients:", group_train[group_col].nunique())
print("Test clients:", group_test[group_col].nunique())
print("Client overlap: none")

,validation_design,precision_at_20,average_precision,roc_auc
0,Random page-level split,0.55,0.636,0.629
1,Client-grouped split,0.50,0.529,0.533


Training clients: 25
Test clients: 7
Client overlap: none


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*



The label is created from `trend_direction`. Therefore, I exclude `trend_pct`, because it directly measures the magnitude of the same trend outcome.

I also exclude all last-30-day and previous-30-day comparison columns because those columns are used to measure the trend. I exclude 90-day performance totals from this strict audit model because they include the same recent period as the observed outcome.

Finally, I exclude `content_id` and `client_id` as model features. They are pseudonymized identifiers, not meaningful page characteristics.

This is a stricter model than my earlier experiment. Its performance may be lower, but its inputs are more appropriate for an honest decision-time claim.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
forbidden_features = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]

audit = pd.DataFrame({
    "column": forbidden_features,
    "used_as_feature": [
        column in feature_cols for column in forbidden_features
    ],
})

audit["reason"] = [
    "Identifier only",
    "Grouping field only",
    "Outcome label",
    "Direct outcome magnitude",
    "Overlaps with recent outcome period",
    "Overlaps with recent outcome period",
    "Overlaps with recent outcome period",
    "Overlaps with recent outcome period",
    "Overlaps with recent outcome period",
    "Overlaps with recent outcome period",
    "Overlaps with recent outcome period",
    "Overlaps with recent outcome period",
    "Outcome-window input",
    "Outcome-window input",
    "Outcome-window input",
    "Comparison-window input",
    "Comparison-window input",
    "Comparison-window input",
]

display(audit)

assert not any(audit["used_as_feature"]), "Leakage audit failed."

print("Leakage audit passed.")
print("Features retained:", feature_cols)

,column,used_as_feature,reason
0,content_id,False,Identifier only
1,client_id,False,Grouping field only
2,trend_direction,False,Outcome label
3,trend_pct,False,Direct outcome magnitude
4,impressions_90d,False,Overlaps with recent outcome period
5,clicks_90d,False,Overlaps with recent outcome period
6,pageviews_90d,False,Overlaps with recent outcome period
7,sessions_90d,False,Overlaps with recent outcome period
8,users_90d,False,Overlaps with recent outcome period
9,engaged_sessions_90d,False,Overlaps with recent outcome period


Leakage audit passed.
Features retained: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'competition_level', 'content_type', 'main_intent']


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*



## Earlier claim

“The model identifies pages that should be refreshed.”

## Revised claim

“Using a client-grouped validation split, the model measured which pages were more likely to have an observed downward trend from stable content and market signals. The resulting score can support a human review queue, but it does not show that refreshing a specific page will improve performance.”

This revised claim is narrower and more honest. It describes an observed ranking pattern, not a causal refresh effect.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
group_results = group_test[
    [id_col, group_col, target_col]
].copy()

group_results["model_score"] = group_scores
group_results["predicted_down"] = (
    group_results["model_score"] >= 0.50
).astype(int)

false_positives = (
    group_results[
        (group_results["predicted_down"] == 1)
        & (group_results[target_col] == 0)
    ]
    .sort_values("model_score", ascending=False)
    .head(10)
)

false_negatives = (
    group_results[
        (group_results["predicted_down"] == 0)
        & (group_results[target_col] == 1)
    ]
    .sort_values("model_score")
    .head(10)
)

print("False positives: predicted down, but observed trend was not down.")
display(false_positives)

print("False negatives: observed down, but model score was below 0.50.")
display(false_negatives)

False positives: predicted down, but observed trend was not down.


,content_id,client_id,decline_label,model_score,predicted_down
10870,content_a5dbb404bdc2,client_f369cb89fc,0,0.704164,1
8521,content_0ea80678706d,client_f369cb89fc,0,0.701394,1
27993,content_26d48a980581,client_f369cb89fc,0,0.697199,1
19045,content_94feac677ec4,client_f369cb89fc,0,0.696458,1
14300,content_24b8d8860ad9,client_8527a891e2,0,0.695783,1
27709,content_84c5f6af423c,client_8527a891e2,0,0.694581,1
18854,content_217ab13a19c2,client_8527a891e2,0,0.694012,1
7646,content_661e0e54bdd8,client_8527a891e2,0,0.693606,1
12439,content_624b2fd8dca5,client_8527a891e2,0,0.693066,1
7248,content_1e445ab8e155,client_8527a891e2,0,0.692564,1


False negatives: observed down, but model score was below 0.50.


,content_id,client_id,decline_label,model_score,predicted_down
8963,content_fd7153335627,client_4e07408562,1,0.268948,0
12500,content_b99427238b18,client_4e07408562,1,0.270102,0
23523,content_38b9529bf1b6,client_e629fa6598,1,0.270663,0
29651,content_7fe7914ad0e0,client_4e07408562,1,0.271559,0
9250,content_3e46637d13e0,client_e629fa6598,1,0.272681,0
14353,content_076e8dccff24,client_e629fa6598,1,0.273479,0
8226,content_19e8e0ca29e9,client_e629fa6598,1,0.273941,0
29374,content_a534caf61146,client_e629fa6598,1,0.275768,0
13922,content_420c7df3a60e,client_e629fa6598,1,0.275908,0
2787,content_604700f57416,client_e629fa6598,1,0.275908,0


## Error interpretation

False positives may be pages with content and market signals associated with decline that remained stable in the observed window. False negatives may be pages whose decline was driven by factors absent from the model, such as seasonality, search-intent changes, client strategy, or competition.

These are measured errors in this dataset. They do not establish that the model is correct or incorrect for every individual page, and they do not prove refresh impact.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.